In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

In [3]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
df = pd.read_csv(PROCESSED_DIR / "sdss_clean.csv")
df.shape

(150000, 18)

In [4]:
features = [
    "dered_u", "dered_g", "dered_r", "dered_i", "dered_z",
    "u_g", "g_r", "r_i", "i_z"
]
target = "object_class"

X = df[features]
y = df[target]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((120000, 9), (30000, 9))

In [6]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
print(classification_report(y_test, dummy.predict(X_test)))

c:\uni\seriousism\Redshifted\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\uni\seriousism\Redshifted\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

      GALAXY       0.33      1.00      0.50     10000
         QSO       0.00      0.00      0.00     10000
        STAR       0.00      0.00      0.00     10000

    accuracy                           0.33     30000
   macro avg       0.11      0.33      0.17     30000
weighted avg       0.11      0.33      0.17     30000



c:\uni\seriousism\Redshifted\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Baseline: predicting the majority class alone gives 33% accuracy (as expected
with 3 balanced classes). Any real model needs to clear this bar by a wide
margin to be considered useful.

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train)
print(classification_report(y_test, logreg.predict(X_test_scaled)))

              precision    recall  f1-score   support

      GALAXY       0.68      0.74      0.71     10000
         QSO       0.78      0.90      0.84     10000
        STAR       0.69      0.52      0.59     10000

    accuracy                           0.72     30000
   macro avg       0.72      0.72      0.72     30000
weighted avg       0.72      0.72      0.72     30000



Logistic Regression reaches 72% accuracy, a big jump from baseline, but STAR
recall is notably weak (0.52), meaning nearly half of STAR objects are
misclassified. This lines up with the EDA finding that STAR and GALAXY overlap
heavily in color space; a linear decision boundary struggles to separate them.

In [8]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test)))

              precision    recall  f1-score   support

      GALAXY       0.91      0.92      0.92     10000
         QSO       0.92      0.94      0.93     10000
        STAR       0.92      0.90      0.91     10000

    accuracy                           0.92     30000
   macro avg       0.92      0.92      0.92     30000
weighted avg       0.92      0.92      0.92     30000



Random Forest jumps to 92% accuracy, with STAR recall recovering to 0.90,
far better than Logistic Regression. This suggests the class boundary between
STAR and GALAXY is non-linear: a model that can learn feature interactions
picks up separating patterns that a linear model misses.